<a href="https://colab.research.google.com/github/ashikjoel/-Context-Aware-Neural-Recommendation-Engine/blob/ashik_joel/week_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import drive

drive.mount('/content/drive')

print("Google Drive mounted successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!


In [2]:
import os

BASE_PATH = "/content/drive/MyDrive/Recommendation_Engine"
DATA_PATH = f"{BASE_PATH}/data/processed"
FEATURE_PATH = f"{BASE_PATH}/features"
VOCAB_PATH = f"{BASE_PATH}/vocabularies"
MODEL_PATH = f"{BASE_PATH}/models"

print("Base path exists   :", os.path.exists(BASE_PATH))
print("Data path exists   :", os.path.exists(DATA_PATH))
print("Feature path exists:", os.path.exists(FEATURE_PATH))
print("Vocab path exists  :", os.path.exists(VOCAB_PATH))
print("Model path exists  :", os.path.exists(MODEL_PATH))

print("\nData files:")
print(os.listdir(DATA_PATH))

print("\nVocabulary files:")
print(os.listdir(VOCAB_PATH))

Base path exists   : True
Data path exists   : True
Feature path exists: True
Vocab path exists  : True
Model path exists  : True

Data files:
['customers_clean.parquet', 'articles_clean.parquet', 'transactions_clean.parquet', 'cold_start_users.parquet', 'cold_start_items.parquet']

Vocabulary files:
['customer_vocab.parquet', 'article_vocab.parquet', 'product_type_vocab.parquet', 'department_vocab.parquet', 'color_vocab.parquet']


In [3]:
import pandas as pd

transactions_path = f"{DATA_PATH}/transactions_clean.parquet"

transactions_check = pd.read_parquet(
    transactions_path,
    engine="pyarrow"
)

print("Rows:", len(transactions_check))
print("\nColumns:")
print(transactions_check.columns.tolist())

print("\nData types:")
print(transactions_check.dtypes)

print("\nFirst 5 rows:")
display(transactions_check.head())

Rows: 31788324

Columns:
['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']

Data types:
t_dat                object
customer_id          object
article_id            int32
price               float64
sales_channel_id      int32
dtype: object

First 5 rows:


,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RecommendationEngine-Week2")
    .getOrCreate()
)

print("Spark version:", spark.version)

transactions = spark.read.parquet(
    f"{DATA_PATH}/transactions_clean.parquet"
)

print("Transaction rows:", transactions.count())

transactions.printSchema()

Spark version: 4.0.4
Transaction rows: 31788324
root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)



In [5]:
from pyspark.sql import functions as F

date_check = transactions.select(
    F.min("t_dat").alias("earliest_date"),
    F.max("t_dat").alias("latest_date")
)

date_check.show()

+-------------+-----------+
|earliest_date|latest_date|
+-------------+-----------+
|   2018-09-20| 2020-09-22|
+-------------+-----------+



In [6]:
transactions.groupBy(
    F.year("t_dat").alias("year"),
    F.month("t_dat").alias("month")
).count().orderBy(
    "year", "month"
).show(30, truncate=False)

+----+-----+-------+
|year|month|count  |
+----+-----+-------+
|2018|9    |594776 |
|2018|10   |1397040|
|2018|11   |1270619|
|2018|12   |1148827|
|2019|1    |1263471|
|2019|2    |1152412|
|2019|3    |1286750|
|2019|4    |1476454|
|2019|5    |1560319|
|2019|6    |1906202|
|2019|7    |1807494|
|2019|8    |1253530|
|2019|9    |1227178|
|2019|10   |1146772|
|2019|11   |1198033|
|2019|12   |1118315|
|2020|1    |1076354|
|2020|2    |1001859|
|2020|3    |1047752|
|2020|4    |1340882|
|2020|5    |1361815|
|2020|6    |1764507|
|2020|7    |1351502|
|2020|8    |1237192|
|2020|9    |798269 |
+----+-----+-------+



In [7]:
test_start = F.date_sub(F.lit("2020-09-22"), 6)

print("Test start :", test_start)
print("Test end   : 2020-09-22")

transactions.filter(
    F.col("t_dat") >= test_start
).select(
    F.min("t_dat").alias("first_test_date"),
    F.max("t_dat").alias("last_test_date"),
    F.count("*").alias("test_rows")
).show()

Test start : Column<'date_sub('2020-09-22', 6)'>
Test end   : 2020-09-22
+---------------+--------------+---------+
|first_test_date|last_test_date|test_rows|
+---------------+--------------+---------+
|     2020-09-16|    2020-09-22|   240311|
+---------------+--------------+---------+



In [8]:
TEST_START = "2020-09-16"

train_transactions = transactions.filter(
    F.col("t_dat") < TEST_START
)

test_transactions = transactions.filter(
    F.col("t_dat") >= TEST_START
)

print("Training transactions:", train_transactions.count())
print("Testing transactions :", test_transactions.count())

Training transactions: 31548013
Testing transactions : 240311


In [9]:
user_history = (
    train_transactions
    .groupBy("customer_id")
    .count()
)

user_history_stats = user_history.select(
    F.min("count").alias("min"),
    F.expr("percentile(count, 0.10)").alias("p10"),
    F.expr("percentile(count, 0.25)").alias("p25"),
    F.expr("percentile(count, 0.50)").alias("median"),
    F.expr("percentile(count, 0.75)").alias("p75"),
    F.expr("percentile(count, 0.90)").alias("p90"),
    F.max("count").alias("max")
)

user_history_stats.show()

+---+---+---+------+----+----+----+
|min|p10|p25|median| p75| p90| max|
+---+---+---+------+----+----+----+
|  1|2.0|3.0|   9.0|27.0|59.0|1895|
+---+---+---+------+----+----+----+



In [10]:
print("Unique training customers:", user_history.count())

print(
    "Unique training interactions:",
    train_transactions
        .select("customer_id", "article_id")
        .dropDuplicates()
        .count()
)

Unique training customers: 1356709
Unique training interactions: 27101148


In [11]:
estimated_sample = (
    user_history
    .select(
        F.sum(
            F.least(F.col("count"), F.lit(10))
        ).alias("estimated_rows")
    )
)

estimated_sample.show()

+--------------+
|estimated_rows|
+--------------+
|       9403922|
+--------------+



In [12]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Remove repeated customer-item interactions first
unique_train = (
    train_transactions
    .select("t_dat", "customer_id", "article_id")
    .dropDuplicates(["customer_id", "article_id"])
)

# Rank each customer's items by most recent interaction
customer_window = Window.partitionBy("customer_id").orderBy(
    F.col("t_dat").desc()
)

# Keep the 10 most recent unique items per customer
train_sample_v2 = (
    unique_train
    .withColumn("rank", F.row_number().over(customer_window))
    .filter(F.col("rank") <= 10)
    .drop("rank")
)

print("Customer-aware training sample created.")
print("Sample rows:", train_sample_v2.count())

Customer-aware training sample created.
Sample rows: 9080208


In [13]:
sample_user_history = (
    train_sample_v2
    .groupBy("customer_id")
    .count()
)

sample_stats = sample_user_history.select(
    F.min("count").alias("min"),
    F.expr("percentile(count, 0.10)").alias("p10"),
    F.expr("percentile(count, 0.25)").alias("p25"),
    F.expr("percentile(count, 0.50)").alias("median"),
    F.expr("percentile(count, 0.75)").alias("p75"),
    F.expr("percentile(count, 0.90)").alias("p90"),
    F.max("count").alias("max")
)

sample_stats.show()

+---+---+---+------+----+----+---+
|min|p10|p25|median| p75| p90|max|
+---+---+---+------+----+----+---+
|  1|1.0|3.0|   8.0|10.0|10.0| 10|
+---+---+---+------+----+----+---+



In [15]:
evaluation_users = (
    test_transactions
    .select("customer_id")
    .distinct()
)

print("Evaluation users:", evaluation_users.count())

evaluation_users.show(5, truncate=False)

Evaluation users: 68984
+----------------------------------------------------------------+
|customer_id                                                     |
+----------------------------------------------------------------+
|03cbb6ef35c9d7f4d53ee1715f438d495e3d884af5c18ad7740f21be9a02fc4b|
|165bf76a599ced4b996d76af3c3f6638b0a2ae72cfcbc00d949c574fe51bcd39|
|1b55ec366ea2e3e354b76479ca6fa1c23ccb6bedf9fabe0f56c8463cd93f2788|
|1c81804fd80e760f59883f2c40e1ceda64f48a968baa815b48dc3d8795d38ff6|
|21e469c6daee2e5dc1a9850a9840d71dc83d229d7e79a0bc3919239c7375538d|
+----------------------------------------------------------------+
only showing top 5 rows


In [16]:
evaluation_sample_history = (
    train_sample_v2
    .join(
        evaluation_users,
        on="customer_id",
        how="inner"
    )
    .groupBy("customer_id")
    .count()
)

evaluation_sample_stats = evaluation_sample_history.select(
    F.min("count").alias("min"),
    F.expr("percentile(count, 0.25)").alias("p25"),
    F.expr("percentile(count, 0.50)").alias("median"),
    F.expr("percentile(count, 0.75)").alias("p75"),
    F.expr("percentile(count, 0.90)").alias("p90"),
    F.max("count").alias("max")
)

print(
    "Evaluation users represented:",
    evaluation_sample_history.count()
)

evaluation_sample_stats.show()

Evaluation users represented: 63412
+---+----+------+----+----+---+
|min| p25|median| p75| p90|max|
+---+----+------+----+----+---+
|  1|10.0|  10.0|10.0|10.0| 10|
+---+----+------+----+----+---+



In [17]:
cold_start_eval_users = (
    evaluation_users
    .join(
        train_sample_v2.select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Cold-start evaluation users:",
    cold_start_eval_users.count()
)

Cold-start evaluation users: 5572


In [18]:
test_user_items = (
    test_transactions
    .select("customer_id", "article_id")
    .dropDuplicates()
)

train_user_items = (
    train_sample_v2
    .select("customer_id", "article_id")
    .dropDuplicates()
)

overlap_count = (
    test_user_items
    .join(
        train_user_items,
        on=["customer_id", "article_id"],
        how="inner"
    )
    .count()
)

total_test_unique = test_user_items.count()

print("Unique test user-item interactions:", total_test_unique)
print("Test items also seen in training   :", overlap_count)
print(
    "Overlap percentage:",
    round(100 * overlap_count / total_test_unique, 2),
    "%"
)

Unique test user-item interactions: 213728
Test items also seen in training   : 5809
Overlap percentage: 2.72 %


In [19]:
customer_vocab_df = spark.read.parquet(
    f"{VOCAB_PATH}/customer_vocab.parquet"
)

article_vocab_df = spark.read.parquet(
    f"{VOCAB_PATH}/article_vocab.parquet"
)

print("Customer vocabulary:")
customer_vocab_df.printSchema()

print("\nArticle vocabulary:")
article_vocab_df.printSchema()

print("\nCustomer vocabulary size:",
      customer_vocab_df.count())

print("Article vocabulary size:",
      article_vocab_df.count())

print("\nFirst 5 customers:")
customer_vocab_df.show(5, truncate=False)

print("First 5 articles:")
article_vocab_df.show(5)

Customer vocabulary:
root
 |-- customer_id: string (nullable = true)


Article vocabulary:
root
 |-- article_id: integer (nullable = true)


Customer vocabulary size: 1371980
Article vocabulary size: 105542

First 5 customers:
+----------------------------------------------------------------+
|customer_id                                                     |
+----------------------------------------------------------------+
|7f8b9b0a806fac06c6bac8bd3aa516591560e04f4d7b1b69a46f9a8208400a51|
|7f8b9ef211ea02981d0576657e49720dc4860d096a527d6bd23879688ec533da|
|7f8ba63d7266bec3428e8b00b9e1aa82e89c86a473235495f0047604064ef44d|
|7f8bb6c6bac4db37c686e72db68330e049f7dbd4d9d94f99db8f7e15fad0345b|
|7f8bdbcae9710dec1d5cfd26c8f506b709cb31592b38a3739ab8b390f28b3967|
+----------------------------------------------------------------+
only showing top 5 rows
First 5 articles:
+----------+
|article_id|
+----------+
| 108775015|
| 108775044|
| 108775051|
| 110065001|
| 110065002|
+----------+
only showin

In [21]:
!pip install -q tensorflow-recommenders==0.7.7

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 5.3 MB/s eta 0:00:00


In [23]:
%env TF_USE_LEGACY_KERAS=1

env: TF_USE_LEGACY_KERAS=1


In [2]:
!pip show tensorflow tensorflow-recommenders keras tf-keras

Name: tensorflow
Version: 2.20.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: absl-py, astunparse, flatbuffers, gast, google_pasta, grpcio, h5py, keras, libclang, ml_dtypes, numpy, opt_einsum, packaging, protobuf, requests, setuptools, six, tensorboard, termcolor, typing_extensions, wrapt
Required-by: dopamine_rl, tensorflow-recommenders, tensorflow-text, tf_keras, ydf_tf
---
Name: tensorflow-recommenders
Version: 0.7.7
Summary: Tensorflow Recommenders, a TensorFlow library for recommender systems.
Home-page: https://github.com/tensorflow/recommenders
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: absl-py, tensorflow, tf-keras
Required-by: 
---
Name: keras
Version: 3.13.2
Summary: Multi-ba

In [3]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

print("TF_USE_LEGACY_KERAS =", os.environ["TF_USE_LEGACY_KERAS"])

TF_USE_LEGACY_KERAS = 1


In [2]:
%env TF_USE_LEGACY_KERAS=1

env: TF_USE_LEGACY_KERAS=1


In [3]:
import os

print("Environment variable:",
      os.environ.get("TF_USE_LEGACY_KERAS"))

import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("tf.keras:", tf.keras.__version__)

Environment variable: 1
TensorFlow: 2.20.0
tf.keras: 3.13.2


In [5]:
!pip index versions tensorflow-recommenders

tensorflow-recommenders (0.7.7)
Available versions: 0.7.7, 0.7.6, 0.7.3, 0.7.2, 0.7.0, 0.6.0, 0.5.2, 0.5.1, 0.5.0, 0.4.0, 0.3.2, 0.3.1, 0.3.0, 0.2.0, 0.1.3, 0.1.2, 0.1.1
  INSTALLED: 0.7.7
  LATEST:    0.7.7


In [10]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RecommendationEngine-Week2")
    .getOrCreate()
)

BASE_PATH = "/content/drive/MyDrive/Recommendation_Engine"
VOCAB_PATH = f"{BASE_PATH}/vocabularies"

customer_vocab_df = spark.read.parquet(
    f"{VOCAB_PATH}/customer_vocab.parquet"
)

article_vocab_df = spark.read.parquet(
    f"{VOCAB_PATH}/article_vocab.parquet"
)

print("Customer vocabulary:", customer_vocab_df.count())
print("Article vocabulary:", article_vocab_df.count())

Customer vocabulary: 1371980
Article vocabulary: 105542


In [11]:
import tensorflow as tf

# Convert Spark vocabularies to Python lists
customer_vocab = [
    row.customer_id
    for row in customer_vocab_df.collect()
]

article_vocab = [
    str(row.article_id)
    for row in article_vocab_df.collect()
]

print("Customer vocabulary:", len(customer_vocab))
print("Article vocabulary:", len(article_vocab))

# Create lookup layers
customer_lookup = tf.keras.layers.StringLookup(
    vocabulary=customer_vocab,
    mask_token=None
)

article_lookup = tf.keras.layers.StringLookup(
    vocabulary=article_vocab,
    mask_token=None
)

print("Customer lookup size:", customer_lookup.vocabulary_size())
print("Article lookup size:", article_lookup.vocabulary_size())

# Verify actual lookup
test_customer = tf.constant([customer_vocab[0]])
test_article = tf.constant([article_vocab[0]])

print("Test customer index:", customer_lookup(test_customer).numpy())
print("Test article index:", article_lookup(test_article).numpy())

Customer vocabulary: 1371980
Article vocabulary: 105542
Customer lookup size: 1371981
Article lookup size: 105543
Test customer index: [1]
Test article index: [1]


In [12]:
EMBEDDING_DIM = 64

customer_embedding = tf.keras.layers.Embedding(
    input_dim=customer_lookup.vocabulary_size(),
    output_dim=EMBEDDING_DIM,
    name="customer_embedding"
)

article_embedding = tf.keras.layers.Embedding(
    input_dim=article_lookup.vocabulary_size(),
    output_dim=EMBEDDING_DIM,
    name="article_embedding"
)

# Test embeddings
customer_test_embedding = customer_embedding(
    customer_lookup(test_customer)
)

article_test_embedding = article_embedding(
    article_lookup(test_article)
)

print("Customer embedding shape:",
      customer_test_embedding.shape)

print("Article embedding shape:",
      article_test_embedding.shape)

Customer embedding shape: (1, 64)
Article embedding shape: (1, 64)


In [13]:
# Query Tower
query_tower = tf.keras.Sequential(
    [
        customer_lookup,
        customer_embedding,
        tf.keras.layers.Dense(
            EMBEDDING_DIM,
            activation="relu",
            name="query_projection"
        )
    ],
    name="query_tower"
)

# Candidate Tower
candidate_tower = tf.keras.Sequential(
    [
        article_lookup,
        article_embedding,
        tf.keras.layers.Dense(
            EMBEDDING_DIM,
            activation="relu",
            name="candidate_projection"
        )
    ],
    name="candidate_tower"
)

# Test both towers
query_test = query_tower(
    tf.constant([customer_vocab[0]])
)

candidate_test = candidate_tower(
    tf.constant([article_vocab[0]])
)

print("Query output shape    :", query_test.shape)
print("Candidate output shape:", candidate_test.shape)

print("\nQuery tower:")
query_tower.summary()

print("\nCandidate tower:")
candidate_tower.summary()

Query output shape    : (1, 64)
Candidate output shape: (1, 64)

Query tower:


Model: "query_tower"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ string_lookup (StringLookup)    │ (1)                    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ customer_embedding (Embedding)  │ (1, 64)                │    87,806,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ query_projection (Dense)        │ (1, 64)                │         4,160 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 87,810,944 (334.97 MB)

 Trainable params: 87,810,944 (334.97 MB)

...

Model: "candidate_tower"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ string_lookup_1 (StringLookup)  │ (1)                    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ article_embedding (Embedding)   │ (1, 64)                │     6,754,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ candidate_projection (Dense)    │ (1, 64)                │         4,160 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,758,912 (25.78 MB)

 Trainable params: 6,758,912 (25.78 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
import tensorflow as tf

# Small sanity-check batch
test_customers = tf.constant([
    str(customer_vocab[0]),
    str(customer_vocab[1]),
    str(customer_vocab[2]),
    str(customer_vocab[3]),
])

test_articles = tf.constant([
    str(article_vocab[0]),
    str(article_vocab[1]),
    str(article_vocab[2]),
    str(article_vocab[3]),
])

# Forward pass
query_vectors = query_tower(test_customers)
candidate_vectors = candidate_tower(test_articles)

# In-batch dot-product scores
scores = tf.matmul(
    query_vectors,
    candidate_vectors,
    transpose_b=True
)

# Each row's diagonal is the positive pair
labels = tf.range(tf.shape(scores)[0])

# Softmax cross-entropy
loss = tf.reduce_mean(
    tf.keras.losses.sparse_categorical_crossentropy(
        labels,
        scores,
        from_logits=True
    )
)

print("Query vectors shape    :", query_vectors.shape)
print("Candidate vectors shape:", candidate_vectors.shape)
print("Score matrix shape     :", scores.shape)
print("Initial loss           :", float(loss.numpy()))
print("Loss is finite         :", bool(tf.math.is_finite(loss).numpy()))

Query vectors shape    : (4, 64)
Candidate vectors shape: (4, 64)
Score matrix shape     : (4, 4)
Initial loss           : 1.3859269618988037
Loss is finite         : True


In [15]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

with tf.GradientTape() as tape:
    query_vectors = query_tower(test_customers, training=True)
    candidate_vectors = candidate_tower(test_articles, training=True)

    scores = tf.matmul(
        query_vectors,
        candidate_vectors,
        transpose_b=True
    )

    labels = tf.range(tf.shape(scores)[0])

    loss_before_update = tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(
            labels,
            scores,
            from_logits=True
        )
    )

# Compute gradients
trainable_variables = (
    query_tower.trainable_variables
    + candidate_tower.trainable_variables
)

gradients = tape.gradient(
    loss_before_update,
    trainable_variables
)

# Check gradients
valid_gradients = [
    g for g in gradients
    if g is not None
]

print("Trainable variables:", len(trainable_variables))
print("Gradients computed  :", len(valid_gradients))

# Apply one update
optimizer.apply_gradients(
    zip(gradients, trainable_variables)
)

# Calculate loss after update
query_vectors_after = query_tower(test_customers, training=False)
candidate_vectors_after = candidate_tower(test_articles, training=False)

scores_after = tf.matmul(
    query_vectors_after,
    candidate_vectors_after,
    transpose_b=True
)

loss_after_update = tf.reduce_mean(
    tf.keras.losses.sparse_categorical_crossentropy(
        labels,
        scores_after,
        from_logits=True
    )
)

print("Loss before update:", float(loss_before_update.numpy()))
print("Loss after update :", float(loss_after_update.numpy()))
print(
    "Loss decreased    :",
    float(loss_after_update.numpy()) < float(loss_before_update.numpy())
)

Trainable variables: 6
Gradients computed  : 6
Loss before update: 1.3859269618988037
Loss after update : 1.3840829133987427
Loss decreased    : True


In [17]:
import os

BASE_PATH = "/content/drive/MyDrive/Recommendation_Engine"

for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        if "sample" in file.lower() or "train" in file.lower():
            print(os.path.join(root, file))

/content/drive/MyDrive/Recommendation_Engine/models/week2_trained/query_tower_trained.keras
/content/drive/MyDrive/Recommendation_Engine/models/week2_trained/query_tower_trained.weights.h5
/content/drive/MyDrive/Recommendation_Engine/models/week2_trained/candidate_tower_trained.weights.h5


In [20]:
import os

BASE_PATH = "/content/drive/MyDrive/Recommendation_Engine"

for root, dirs, files in os.walk(BASE_PATH):
    if "transactions_clean.parquet" in files:
        print("FOUND:")
        print(os.path.join(root, "transactions_clean.parquet"))

In [21]:
import os

BASE_PATH = "/content/drive/MyDrive/Recommendation_Engine"

print("Base exists:", os.path.exists(BASE_PATH))
print("\nContents:")

for item in os.listdir(BASE_PATH):
    print(item)

Base exists: True

Contents:
data
features
vocabularies
models
final_retrieval_index


In [22]:
DATA_PATH = "/content/drive/MyDrive/Recommendation_Engine/data"

print("Data folder exists:", os.path.exists(DATA_PATH))
print("\nFiles/folders inside data:")

for item in os.listdir(DATA_PATH):
    print(item)

Data folder exists: True

Files/folders inside data:
processed


In [23]:
PROCESSED_PATH = "/content/drive/MyDrive/Recommendation_Engine/data/processed"

print("Processed folder exists:", os.path.exists(PROCESSED_PATH))
print("\nContents:")

for item in os.listdir(PROCESSED_PATH):
    print(item)

Processed folder exists: True

Contents:
customers_clean.parquet
articles_clean.parquet
transactions_clean.parquet
cold_start_users.parquet
cold_start_items.parquet


In [24]:
from pyspark.sql import functions as F

TRANSACTIONS_PATH = (
    "/content/drive/MyDrive/Recommendation_Engine/"
    "data/processed/transactions_clean.parquet"
)

transactions = spark.read.parquet(TRANSACTIONS_PATH)

print("Transaction rows:", transactions.count())

transactions.printSchema()

transactions.select(
    "t_dat",
    "customer_id",
    "article_id"
).show(5, truncate=False)

Transaction rows: 31788324
root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)

+----------+----------------------------------------------------------------+----------+
|t_dat     |customer_id                                                     |article_id|
+----------+----------------------------------------------------------------+----------+
|2019-11-29|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016003 |
|2019-11-29|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016001 |
|2019-11-29|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|682236013 |
|2019-11-29|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016016 |
|2019-11-29|aaa7a0483dd5b9e395d95324dcbfeb617af9800f39487d4b6aaee662bcd384c7|783335003 |
+----------+------------------------------------

In [25]:
# Test period: last 7 days
test_end = transactions.agg(
    F.max("t_dat")
).collect()[0][0]

test_start = F.date_sub(F.lit(test_end), 6)

print("Test start:", test_start)
print("Test end  :", test_end)

# Training data = everything before test period
train_transactions = transactions.filter(
    F.col("t_dat") < test_start
)

test_transactions = transactions.filter(
    F.col("t_dat") >= test_start
)

print("Training transactions:", train_transactions.count())
print("Testing transactions :", test_transactions.count())

Test start: Column<'date_sub(2020-09-22, 6)'>
Test end  : 2020-09-22
Training transactions: 31548013
Testing transactions : 240311


In [26]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Keep only unique customer-item interactions.
# For each customer/article pair, retain the most recent purchase.
customer_item_history = (
    train_transactions
    .groupBy("customer_id", "article_id")
    .agg(
        F.max("t_dat").alias("t_dat")
    )
)

print(
    "Unique customer-item interactions:",
    customer_item_history.count()
)

# Rank articles by recency for each customer
window_spec = Window.partitionBy("customer_id").orderBy(
    F.col("t_dat").desc()
)

train_sample_v2 = (
    customer_item_history
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") <= 10)
    .drop("rn")
)

print(
    "Customer-aware training sample:",
    train_sample_v2.count()
)

print("\nHistory distribution:")
train_sample_v2.groupBy("customer_id").count().select(
    F.min("count").alias("min"),
    F.expr("percentile(count, 0.10)").alias("p10"),
    F.expr("percentile(count, 0.25)").alias("p25"),
    F.expr("percentile(count, 0.50)").alias("median"),
    F.expr("percentile(count, 0.75)").alias("p75"),
    F.expr("percentile(count, 0.90)").alias("p90"),
    F.max("count").alias("max")
).show()

Unique customer-item interactions: 27101148
Customer-aware training sample: 9080208

History distribution:
+---+---+---+------+----+----+---+
|min|p10|p25|median| p75| p90|max|
+---+---+---+------+----+----+---+
|  1|1.0|3.0|   8.0|10.0|10.0| 10|
+---+---+---+------+----+----+---+



In [27]:
SAMPLE_PATH = (
    "/content/drive/MyDrive/Recommendation_Engine/"
    "data/training_samples"
)

(
    train_sample_v2
    .write
    .mode("overwrite")
    .parquet(SAMPLE_PATH)
)

print("Training sample saved successfully!")
print("Path:", SAMPLE_PATH)

Training sample saved successfully!
Path: /content/drive/MyDrive/Recommendation_Engine/data/training_samples


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RecommendationEngine-Training")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.4


In [3]:
SAMPLE_PATH = (
    "/content/drive/MyDrive/Recommendation_Engine/"
    "data/training_samples"
)

train_sample_v2 = spark.read.parquet(SAMPLE_PATH)

print("Saved training sample rows:", train_sample_v2.count())
train_sample_v2.printSchema()

Saved training sample rows: 9080208
root
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- t_dat: date (nullable = true)



In [4]:
import tensorflow as tf

VOCAB_PATH = "/content/drive/MyDrive/Recommendation_Engine/vocabularies"

customer_vocab_df = spark.read.parquet(
    f"{VOCAB_PATH}/customer_vocab.parquet"
)

article_vocab_df = spark.read.parquet(
    f"{VOCAB_PATH}/article_vocab.parquet"
)

customer_vocab = [
    row.customer_id
    for row in customer_vocab_df.collect()
]

article_vocab = [
    str(row.article_id)
    for row in article_vocab_df.collect()
]

print("Customers:", len(customer_vocab))
print("Articles :", len(article_vocab))

Customers: 1371980
Articles : 105542


In [5]:
EMBEDDING_DIM = 64

customer_lookup = tf.keras.layers.StringLookup(
    vocabulary=customer_vocab,
    mask_token=None
)

article_lookup = tf.keras.layers.StringLookup(
    vocabulary=article_vocab,
    mask_token=None
)

customer_embedding = tf.keras.layers.Embedding(
    input_dim=customer_lookup.vocabulary_size(),
    output_dim=EMBEDDING_DIM,
    name="customer_embedding"
)

article_embedding = tf.keras.layers.Embedding(
    input_dim=article_lookup.vocabulary_size(),
    output_dim=EMBEDDING_DIM,
    name="article_embedding"
)

query_tower = tf.keras.Sequential(
    [
        customer_lookup,
        customer_embedding,
        tf.keras.layers.Dense(
            EMBEDDING_DIM,
            activation="relu",
            name="query_projection"
        )
    ],
    name="query_tower"
)

candidate_tower = tf.keras.Sequential(
    [
        article_lookup,
        article_embedding,
        tf.keras.layers.Dense(
            EMBEDDING_DIM,
            activation="relu",
            name="candidate_projection"
        )
    ],
    name="candidate_tower"
)

print("Query tower and Candidate tower created successfully.")

Query tower and Candidate tower created successfully.


In [6]:
query_test = query_tower(
    tf.constant([customer_vocab[0]])
)

candidate_test = candidate_tower(
    tf.constant([article_vocab[0]])
)

print("Query output    :", query_test.shape)
print("Candidate output:", candidate_test.shape)

Query output    : (1, 64)
Candidate output: (1, 64)


In [8]:
from pyspark.sql import functions as F

train_pairs = (
    train_sample_v2
    .select("customer_id", "article_id")
    .withColumn(
        "article_id",
        F.col("article_id").cast("string")
    )
)

print("Training pairs:", train_pairs.count())

train_pairs.printSchema()

Training pairs: 9080208
root
 |-- customer_id: string (nullable = true)
 |-- article_id: string (nullable = true)



In [9]:
import tensorflow as tf

BATCH_SIZE = 8192

# Take a small sample only for pipeline testing
sample_batch_df = train_pairs.limit(BATCH_SIZE)

sample_rows = sample_batch_df.collect()

batch_customers = tf.constant(
    [row["customer_id"] for row in sample_rows],
    dtype=tf.string
)

batch_articles = tf.constant(
    [row["article_id"] for row in sample_rows],
    dtype=tf.string
)

print("Customer batch shape:", batch_customers.shape)
print("Article batch shape :", batch_articles.shape)

# Pass through the towers
batch_queries = query_tower(batch_customers)
batch_candidates = candidate_tower(batch_articles)

print("Query vectors shape    :", batch_queries.shape)
print("Candidate vectors shape:", batch_candidates.shape)

Customer batch shape: (8192,)
Article batch shape : (8192,)
Query vectors shape    : (8192, 64)
Candidate vectors shape: (8192, 64)


In [10]:
import tensorflow as tf

BATCH_SIZE = 1024

# Use the first 1024 already-loaded rows
small_df = train_pairs.limit(BATCH_SIZE)
small_rows = small_df.collect()

customers = tf.constant(
    [r["customer_id"] for r in small_rows],
    dtype=tf.string
)

articles = tf.constant(
    [r["article_id"] for r in small_rows],
    dtype=tf.string
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

with tf.GradientTape() as tape:

    query_vectors = query_tower(
        customers,
        training=True
    )

    candidate_vectors = candidate_tower(
        articles,
        training=True
    )

    # In-batch negative scoring
    scores = tf.matmul(
        query_vectors,
        candidate_vectors,
        transpose_b=True
    )

    labels = tf.range(BATCH_SIZE)

    loss = tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(
            labels,
            scores,
            from_logits=True
        )
    )

gradients = tape.gradient(
    loss,
    query_tower.trainable_variables +
    candidate_tower.trainable_variables
)

print("Query vectors    :", query_vectors.shape)
print("Candidate vectors:", candidate_vectors.shape)
print("Score matrix     :", scores.shape)
print("Loss             :", float(loss.numpy()))
print(
    "Loss finite      :",
    bool(tf.math.is_finite(loss).numpy())
)

# Check gradients
valid_gradients = [
    g for g in gradients
    if g is not None
]

print("Gradients:", len(valid_gradients))

# Apply one actual update
optimizer.apply_gradients(
    zip(
        gradients,
        query_tower.trainable_variables +
        candidate_tower.trainable_variables
    )
)

print("One training update completed successfully.")

Query vectors    : (1024, 64)
Candidate vectors: (1024, 64)
Score matrix     : (1024, 1024)
Loss             : 6.931554794311523
Loss finite      : True
Gradients: 6
One training update completed successfully.


In [12]:
# Build the fresh towers with a real input
_ = query_tower(
    tf.constant([customer_vocab[0]], dtype=tf.string)
)

_ = candidate_tower(
    tf.constant([article_vocab[0]], dtype=tf.string)
)

print("Fresh model created and built.")

print("Query parameters    :", query_tower.count_params())
print("Candidate parameters:", candidate_tower.count_params())
print(
    "Total parameters    :",
    query_tower.count_params() + candidate_tower.count_params()
)

# Verify output dimensions
q_test = query_tower(
    tf.constant([customer_vocab[0]], dtype=tf.string)
)

c_test = candidate_tower(
    tf.constant([article_vocab[0]], dtype=tf.string)
)

print("Query output    :", q_test.shape)
print("Candidate output:", c_test.shape)

Fresh model created and built.
Query parameters    : 87810944
Candidate parameters: 6758912
Total parameters    : 94569856
Query output    : (1, 64)
Candidate output: (1, 64)


In [13]:
import os
import json
import tensorflow as tf

MODEL_DIR = (
    "/content/drive/MyDrive/Recommendation_Engine/"
    "models/best_model"
)

CHECKPOINT_DIR = os.path.join(
    MODEL_DIR,
    "checkpoints"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 1024
EPOCHS = 3
LEARNING_RATE = 0.001

optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

print("Training configuration:")
print("Batch size     :", BATCH_SIZE)
print("Epochs         :", EPOCHS)
print("Learning rate  :", LEARNING_RATE)
print("Model directory:", MODEL_DIR)
print("Checkpoint dir :", CHECKPOINT_DIR)

Training configuration:
Batch size     : 1024
Epochs         : 3
Learning rate  : 0.001
Model directory: /content/drive/MyDrive/Recommendation_Engine/models/best_model
Checkpoint dir : /content/drive/MyDrive/Recommendation_Engine/models/best_model/checkpoints


In [14]:
import time
import tensorflow as tf

NUM_TEST_BATCHES = 100

# Take 100 batches from the saved Spark data
test_rows = (
    train_pairs
    .limit(NUM_TEST_BATCHES * BATCH_SIZE)
    .collect()
)

print("Rows loaded:", len(test_rows))

losses = []
start_time = time.time()

for step in range(NUM_TEST_BATCHES):

    batch_rows = test_rows[
        step * BATCH_SIZE:(step + 1) * BATCH_SIZE
    ]

    customers = tf.constant(
        [r["customer_id"] for r in batch_rows],
        dtype=tf.string
    )

    articles = tf.constant(
        [r["article_id"] for r in batch_rows],
        dtype=tf.string
    )

    with tf.GradientTape() as tape:

        query_vectors = query_tower(
            customers,
            training=True
        )

        candidate_vectors = candidate_tower(
            articles,
            training=True
        )

        scores = tf.matmul(
            query_vectors,
            candidate_vectors,
            transpose_b=True
        )

        labels = tf.range(BATCH_SIZE)

        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                labels,
                scores,
                from_logits=True
            )
        )

    variables = (
        query_tower.trainable_variables
        + candidate_tower.trainable_variables
    )

    gradients = tape.gradient(loss, variables)

    optimizer.apply_gradients(
        zip(gradients, variables)
    )

    loss_value = float(loss.numpy())
    losses.append(loss_value)

    if (step + 1) % 10 == 0:
        print(
            f"Step {step + 1:3d}/{NUM_TEST_BATCHES} "
            f"- loss: {loss_value:.4f}"
        )

elapsed = time.time() - start_time

print("\n--- Smoke Test Results ---")
print("Initial loss:", losses[0])
print("Final loss  :", losses[-1])
print("Minimum loss:", min(losses))
print("Time        :", round(elapsed, 2), "seconds")
print(
    "Loss finite :",
    all(tf.math.is_finite(x).numpy() for x in losses)
)

Rows loaded: 102400
Step  10/100 - loss: 6.9314
Step  20/100 - loss: 6.9314
Step  30/100 - loss: 6.9315
Step  40/100 - loss: 6.9316
Step  50/100 - loss: 6.9316
Step  60/100 - loss: 6.9315
Step  70/100 - loss: 6.9315
Step  80/100 - loss: 6.9315
Step  90/100 - loss: 6.9315
Step 100/100 - loss: 6.9317

--- Smoke Test Results ---
Initial loss: 6.931467533111572
Final loss  : 6.931668758392334
Minimum loss: 6.931331157684326
Time        : 6.75 seconds
Loss finite : True


In [15]:
# Reset both towers again so this experiment starts from fresh weights

customer_embedding = tf.keras.layers.Embedding(
    input_dim=customer_lookup.vocabulary_size(),
    output_dim=EMBEDDING_DIM,
    name="customer_embedding"
)

article_embedding = tf.keras.layers.Embedding(
    input_dim=article_lookup.vocabulary_size(),
    output_dim=EMBEDDING_DIM,
    name="article_embedding"
)

query_tower = tf.keras.Sequential([
    customer_lookup,
    customer_embedding,
    tf.keras.layers.Dense(
        EMBEDDING_DIM,
        activation="relu",
        name="query_projection"
    )
], name="query_tower")

candidate_tower = tf.keras.Sequential([
    article_lookup,
    article_embedding,
    tf.keras.layers.Dense(
        EMBEDDING_DIM,
        activation="relu",
        name="candidate_projection"
    )
], name="candidate_tower")

# Build
_ = query_tower(tf.constant([customer_vocab[0]], dtype=tf.string))
_ = candidate_tower(tf.constant([article_vocab[0]], dtype=tf.string))

# Lower learning rate for controlled experiment
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0001
)

print("Fresh model ready.")
print("Learning rate:", 0.0001)
print("Query params:", query_tower.count_params())
print("Candidate params:", candidate_tower.count_params())

Fresh model ready.
Learning rate: 0.0001
Query params: 87810944
Candidate params: 6758912


In [16]:
import time

NUM_TEST_BATCHES = 500
BATCH_SIZE = 1024

# Load only the rows needed for this experiment
experiment_rows = (
    train_pairs
    .limit(NUM_TEST_BATCHES * BATCH_SIZE)
    .collect()
)

print("Rows loaded:", len(experiment_rows))

losses = []
start_time = time.time()

variables = (
    query_tower.trainable_variables +
    candidate_tower.trainable_variables
)

for step in range(NUM_TEST_BATCHES):

    batch_rows = experiment_rows[
        step * BATCH_SIZE:(step + 1) * BATCH_SIZE
    ]

    customers = tf.constant(
        [r["customer_id"] for r in batch_rows],
        dtype=tf.string
    )

    articles = tf.constant(
        [r["article_id"] for r in batch_rows],
        dtype=tf.string
    )

    with tf.GradientTape() as tape:

        query_vectors = query_tower(
            customers,
            training=True
        )

        candidate_vectors = candidate_tower(
            articles,
            training=True
        )

        scores = tf.matmul(
            query_vectors,
            candidate_vectors,
            transpose_b=True
        )

        labels = tf.range(BATCH_SIZE)

        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                labels,
                scores,
                from_logits=True
            )
        )

    gradients = tape.gradient(loss, variables)

    optimizer.apply_gradients(
        zip(gradients, variables)
    )

    loss_value = float(loss.numpy())
    losses.append(loss_value)

    if (step + 1) % 50 == 0:
        print(
            f"Step {step + 1:3d}/{NUM_TEST_BATCHES} "
            f"- loss: {loss_value:.5f}"
        )

elapsed = time.time() - start_time

print("\n--- 500-Step Experiment ---")
print("Initial loss:", losses[0])
print("Final loss  :", losses[-1])
print("Minimum loss:", min(losses))
print("Time        :", round(elapsed, 2), "seconds")
print(
    "Loss finite:",
    all(tf.math.is_finite(x).numpy() for x in losses)
)

Rows loaded: 512000
Step  50/500 - loss: 6.93157
Step 100/500 - loss: 6.93149
Step 150/500 - loss: 6.93146
Step 200/500 - loss: 6.93148
Step 250/500 - loss: 6.93151
Step 300/500 - loss: 6.93153
Step 350/500 - loss: 6.93141
Step 400/500 - loss: 6.93174
Step 450/500 - loss: 6.93154
Step 500/500 - loss: 6.93144

--- 500-Step Experiment ---
Initial loss: 6.931487560272217
Final loss  : 6.931439399719238
Minimum loss: 6.931265830993652
Time        : 34.38 seconds
Loss finite: True
